# Fine-Tuning a Small LLM for Tool Routing

**Goal:** Train a 1B parameter model to correctly route HR queries to the right tool.

| Query Type | Tool | Example |
|---|---|---|
| Leave balance enquiry| `check_leave_balance`| "Hey, how many casual leaves do I have left?"|
| Applying for leave| `apply_leave` | "I need a sick leave tomorrow, please apply it for me." |

**What we'll show:**
1. The base model **cannot** do structured tool routing
2. We create a training dataset (~80 examples)
3. Fine-tune with Unsloth + LoRA in ~10 minutes on free Colab T4
4. The fine-tuned model **nails it** — correct tool, correct params, valid JSON

## Step 0: Install Unsloth

In [8]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## Step 1: Load the Base Model

We use **Llama 3.2 1B Instruct** in 4-bit quantization.

Why 1B? It's small enough to train fast on free Colab, and small enough that it will clearly **fail** at structured tool calling — making the before/after dramatic.

In [9]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
print("Model loaded.")

==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.


Model loaded.


## Step 2: Define Our Tools and Test the Base Model

We define two tools that a customer support system would use.
Then we ask the base model to route queries — and watch it fail.

In [10]:
SYSTEM_PROMPT = """You are a Human resource query support assistant. You have access to exactly five tools:

1. check_leave_balance: Call this tool when the employee wants to know how many leaves they have remaining.
   Triggers include questions about leave count, remaining days, available leaves, or balance for any leave type — casual, sick, or earned.
   Parameters: {"leave_type": string}

2. apply_leave: Call this tool when the employee wants to request or apply for a leave for specific dates.
   Triggers include requests to book, apply, or take a leave on a particular day or date range.
   Parameters: {"leave_type": string, "from_date": date , "to_date": date}

3. get_payslip: Call this tool when the employee wants to view or download their salary slip for a specific month.
   Triggers include mentions of payslip, salary slip, or salary details for a given month or year.
   Parameters: {"month": string, "year": string}

4. raise_ticket: Call this tool when the employee is reporting a problem, complaint, or issue that needs HR attention and resolution.
   Triggers include complaints about payroll errors, reimbursement issues, policy concerns, or any unresolved HR matter.
   Parameters: {"category": string, "description": string}

5. get_company_policy: Call this tool when the employee is asking a question about company rules, policies, or entitlements — without requesting a specific action.
   Triggers include questions about leave rules, work from home policy, notice period, reimbursement limits, or any HR guideline.
   Parameters: {"topic": string}


RESPOND WITH ONLY a JSON object in this exact format, nothing else:
{"tool": "<tool_name>", "params": {<parameters>}}

Do not explain. Do not add any text before or after the JSON."""

TEST_QUERIES = [
    # apply_leave — "balance" word but the real intent is booking
    "Forget the balance, just lock in a casual leave for me this Friday.",
    # get_company_policy — sounds like a complaint but is asking the rule
    "Honestly the WFH situation is confusing, what does the policy actually say about remote days?",
    # get_payslip — "salary" mentioned, wants the document not a fix
    "I need my salary slip for January 2026 for a rental agreement.",
    # raise_ticket — "payslip" word but it's a problem report
    "The deductions on my December 2025 payslip don't add up, please get this looked at.",
    # check_leave_balance — no leave_type stated -> general
    "Before I plan anything, how many leaves are even left with me?",
    # apply_leave — date range buried in a story
    "My parents are visiting from the village and I want to be home, block 8th to 11th July as earned leave.",
    # get_company_policy — "how many ... I" phrasing but about entitlement
    "How many sick leaves am I entitled to in a calendar year as per company rules?",
    # check_leave_balance — Hinglish, specific type
    "thoda batao casual leave kitni bachi hai",
    # raise_ticket — PF issue, no obvious keyword beyond context
    "My provident fund contributions stopped showing up two months ago, need someone to act on it.",
    # get_payslip — Hinglish, relative month
    "pichhle mahine ki payslip chahiye thi",
    # apply_leave — single explicit date, sick
    "Not well at all, apply my sick leave for 20th June.",
    # get_company_policy — notice period
    "What's the notice period I'm bound to if I decide to move on?",
    # raise_ticket — reimbursement complaint phrased politely
    "Could someone check why my client-visit reimbursement from last month still hasn't landed?",
    # check_leave_balance — earned, abbreviation
    "How many ELs are sitting in my account right now?",
    # apply_leave — multi-intent, action is primary, policy is secondary
    "Apply casual leave for 25th June for me, and remind me what the holiday list looks like.",
    # get_company_policy — attendance regularization rule
    "If I forget to punch in, how does attendance regularization work here?",
    # raise_ticket — attendance complaint
    "I was physically in office on 3rd June but the system says I was absent.",
    # get_payslip — explicit older month/year
    "Pull my November 2025 pay statement, I need it for tax filing.",
]

#User -> Query -> System (Server) -> Model -> Tool Call Response -> System makes tool call -> System augments query with tool call response -> Model responds to query -> System forwards response to User.

EXPECTED = [
    "apply_leave",
    "get_company_policy",
    "get_payslip",
    "raise_ticket",
    "check_leave_balance",
    "apply_leave",
    "get_company_policy",
    "check_leave_balance",
    "raise_ticket",
    "get_payslip",
    "apply_leave",
    "get_company_policy",
    "raise_ticket",
    "check_leave_balance",
    "apply_leave",
    "get_company_policy",
    "raise_ticket",
    "get_payslip"
]

print("Tools and test queries defined.")

Tools and test queries defined.


In [11]:
# Enable inference mode for the base model
model.eval()

def run_inference(model, tokenizer, user_query):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(
        [text],
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            use_cache=False,
        )
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()


print("=" * 70)
print("BASE MODEL RESPONSES (before fine-tuning)")
print("=" * 70)
for query, expected_tool in zip(TEST_QUERIES, EXPECTED):
    response = run_inference(model, tokenizer, query)
    print(f"\n{'─' * 60}")
    print(f"QUERY:    {query[:80]}")
    print(f"EXPECTED: {expected_tool}")
    print(f"GOT:      {response[:200]}")

    # Check if it's valid JSON with correct tool
    import json
    try:
        parsed = json.loads(response)
        if parsed.get("tool") == expected_tool:
            print("RESULT:   ✅ Correct (lucky!)")
        else:
            print(f"RESULT:   ❌ Wrong tool or format")
    except json.JSONDecodeError:
        print("RESULT:   ❌ Not valid JSON")

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL RESPONSES (before fine-tuning)


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    Forget the balance, just lock in a casual leave for me this Friday.
EXPECTED: apply_leave
GOT:      {"tool": "apply_leave", "params": {"leave_type": "casual", "from_date": "2023-06-03", "to_date": "2023-06-07"}}
RESULT:   ✅ Correct (lucky!)


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


────────────────────────────────────────────────────────────
QUERY:    Apply casual leave for 25th June for me, and remind me what the holiday list loo
EXPECTED: apply_leave
GOT:      {"tool": "apply_leave", "params": {"leave_type": "casual", "from_date": "2023-06-25", "to_date": "2023-06-25"}}

{"tool": "get_payslip", "params": {"month": "June", "year": "2023"}}
RESULT:   ❌ Not valid JSON


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    If I forget to punch in, how does attendance regularization work here?
EXPECTED: get_company_policy
GOT:      {"tool": "get_payslip", "params": {"month": "January", "year": "2024", "from_date": "2024-01-01", "to_date": "2024-01-31"}}
RESULT:   ❌ Wrong tool or format


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    I was physically in office on 3rd June but the system says I was absent.
EXPECTED: raise_ticket
GOT:      {"tool": "check_leave_balance", "params": {"leave_type": "absent", "from_date": "2023-06-03", "to_date": "2023-06-03"}}
RESULT:   ❌ Wrong tool or format

────────────────────────────────────────────────────────────
QUERY:    Pull my November 2025 pay statement, I need it for tax filing.
EXPECTED: get_payslip
GOT:      {"tool": "get_payslip", "params": {"month": "November 2025", "year": "2025"}}
RESULT:   ✅ Correct (lucky!)


### Observation

The 1B instruct model typically:
- Responds in natural language instead of JSON
- Adds preamble text before/after the JSON
- Gets the tool name wrong or hallucinates parameters
- Is inconsistent — sometimes close, sometimes garbage

**This is the problem we're solving with fine-tuning.**

---
## Step 3: Create the Training Dataset

We generate ~80 diverse examples — 40 per tool.
Each example is a conversation: system prompt → user query → correct JSON tool call.

**Key insight for your students:** The dataset IS the product. The model architecture is fixed, the training code is boilerplate. Your engineering judgment lives entirely in the data.

In [12]:
import json
import random

# ──────────────────────────────────────────────
# CHECK_LEAVE_BALANCE examples
# ──────────────────────────────────────────────
check_leave_balance_examples  = [
    ("What's my sick leave balance?", {"leave_type": "sick"}),
    ("How many earned leaves have I got remaining?", {"leave_type": "earned"}),
    ("How many days off do I still have?", {"leave_type": "general"}),
    ("Check my casual leave balance please.", {"leave_type": "casual"}),
    ("Do I have any sick leaves left this year?", {"leave_type": "sick"}),
    ("I want to know my remaining earned leaves.", {"leave_type": "earned"}),
    ("kitni casual leave bachi hai meri?", {"leave_type": "casual"}),
    ("How much leave balance is left in my account?", {"leave_type": "general"}),
    ("Quick check - sick leaves remaining?", {"leave_type": "sick"}),
    ("Tell me my earned leave count.", {"leave_type": "earned"}),
    ("Am I out of casual leaves yet?", {"leave_type": "casual"}),
    ("What is my total available leave?", {"leave_type": "general"}),
    ("How many CLs do I have?", {"leave_type": "casual"}),
    ("SL balance check kar do.", {"leave_type": "sick"}),
    ("How many privilege leaves are pending for me?", {"leave_type": "earned"}),
    ("Just curious how many leaves I can still take.", {"leave_type": "general"}),
    ("Sick leave kitni hai abhi?", {"leave_type": "sick"}),
    ("Do I have enough casual leaves for a long weekend?", {"leave_type": "casual"}),
    ("Show me my earned leave balance.", {"leave_type": "earned"}),
    ("Have all my leaves been used up?", {"leave_type": "general"}),
    ("What's left in my casual leave bucket?", {"leave_type": "casual"}),
    ("How many sick days am I left with?", {"leave_type": "sick"}),
    ("Earned leave balance please.", {"leave_type": "earned"}),
    ("Planning a trip - how many leaves do I have in total?", {"leave_type": "general"}),
    ("Remaining casual leaves?", {"leave_type": "casual"}),
    ("Confirm my sick leave count for this quarter.", {"leave_type": "sick"}),
    ("How many EL do I have accumulated?", {"leave_type": "earned"}),
    ("Give me a summary of all my leave balances.", {"leave_type": "general"}),
    ("bro how many casual leaves left", {"leave_type": "casual"}),
    ("Need my sick leave balance for my records.", {"leave_type": "sick"}),
    ("What's the status of my earned leaves?", {"leave_type": "earned"}),
    ("How many leaves are still in my kitty overall?", {"leave_type": "general"}),
    ("Mere paas kitni earned leave hai?", {"leave_type": "earned"}),
    ("Casual leave balance dikha do.", {"leave_type": "casual"}),
    ("Any sick leaves left or all gone?", {"leave_type": "sick"}),
]

# ──────────────────────────────────────────────
#

# ──────────────────────────────────────────────
# APPLY_LEAVE examples
# ──────────────────────────────────────────────
apply_leave_examples = [
    ("I need a sick leave today.", {"leave_type": "sick", "from_date": "today", "to_date": "today"}),
    ("Apply casual leave for me tomorrow.", {"leave_type": "casual", "from_date": "tomorrow", "to_date": "tomorrow"}),
    ("Book my earned leave from 22nd to 24th June.", {"leave_type": "earned", "from_date": "2026-06-22", "to_date": "2026-06-24"}),
    ("I want to apply for sick leave on 15th July.", {"leave_type": "sick", "from_date": "2026-07-15", "to_date": "2026-07-15"}),
    ("Put me down for casual leave next Monday.", {"leave_type": "casual", "from_date": "next monday", "to_date": "next monday"}),
    ("Apply earned leave for 1st to 5th August please.", {"leave_type": "earned", "from_date": "2026-08-01", "to_date": "2026-08-05"}),
    ("Mark me sick leave for tomorrow and the day after.", {"leave_type": "sick", "from_date": "tomorrow", "to_date": "day after tomorrow"}),
    ("Take a casual leave for me on 12th June.", {"leave_type": "casual", "from_date": "2026-06-12", "to_date": "2026-06-12"}),
    ("I'll be on earned leave from 18th to 20th June, please apply.", {"leave_type": "earned", "from_date": "2026-06-18", "to_date": "2026-06-20"}),
    ("Apply CL for 25th June.", {"leave_type": "casual", "from_date": "2026-06-25", "to_date": "2026-06-25"}),
    ("I want earned leave from 1st to 3rd July.", {"leave_type": "earned", "from_date": "2026-07-01", "to_date": "2026-07-03"}),
    ("kal ki sick leave laga do.", {"leave_type": "sick", "from_date": "tomorrow", "to_date": "tomorrow"}),
    ("Book casual leave for 30th June for me.", {"leave_type": "casual", "from_date": "2026-06-30", "to_date": "2026-06-30"}),
    ("Going on vacation, apply earned leave 10th to 17th July.", {"leave_type": "earned", "from_date": "2026-07-10", "to_date": "2026-07-17"}),
    ("Not feeling well, take sick leave for me on 9th June.", {"leave_type": "sick", "from_date": "2026-06-09", "to_date": "2026-06-09"}),
    ("I need a casual leave on the 14th of June.", {"leave_type": "casual", "from_date": "2026-06-14", "to_date": "2026-06-14"}),
    ("Apply earned leave from 5th to 6th August.", {"leave_type": "earned", "from_date": "2026-08-05", "to_date": "2026-08-06"}),
    ("Sick leave chahiye aaj, please apply.", {"leave_type": "sick", "from_date": "today", "to_date": "today"}),
    ("Apply casual leave for 19th and 20th June.", {"leave_type": "casual", "from_date": "2026-06-19", "to_date": "2026-06-20"}),
    ("My sister's wedding is 22nd to 24th June, I need those three days as earned leave.", {"leave_type": "earned", "from_date": "2026-06-22", "to_date": "2026-06-24"}),
    ("Apply a sick leave for me tomorrow please.", {"leave_type": "sick", "from_date": "tomorrow", "to_date": "tomorrow"}),
    ("I'd like to use a casual leave on 27th June.", {"leave_type": "casual", "from_date": "2026-06-27", "to_date": "2026-06-27"}),
    ("Book earned leave 13th to 16th July.", {"leave_type": "earned", "from_date": "2026-07-13", "to_date": "2026-07-16"}),
    ("Down with fever, mark sick leave today and tomorrow.", {"leave_type": "sick", "from_date": "today", "to_date": "tomorrow"}),
    ("Apply casual leave for 2nd July for me.", {"leave_type": "casual", "from_date": "2026-07-02", "to_date": "2026-07-02"}),
    ("I want to take earned leave on 8th August.", {"leave_type": "earned", "from_date": "2026-08-08", "to_date": "2026-08-08"}),
    ("Please put a sick leave on 11th June.", {"leave_type": "sick", "from_date": "2026-06-11", "to_date": "2026-06-11"}),
    ("Need casual leave from 23rd to 24th June.", {"leave_type": "casual", "from_date": "2026-06-23", "to_date": "2026-06-24"}),
    ("Apply earned leave for all of next week, 15th to 19th June.", {"leave_type": "earned", "from_date": "2026-06-15", "to_date": "2026-06-19"}),
    ("Take a sick leave for me on 28th June please.", {"leave_type": "sick", "from_date": "2026-06-28", "to_date": "2026-06-28"}),
    ("I'll be out on casual leave on 7th July, kindly apply.", {"leave_type": "casual", "from_date": "2026-07-07", "to_date": "2026-07-07"}),
    ("Apply leave for me this Friday, casual.", {"leave_type": "casual", "from_date": "this friday", "to_date": "this friday"}),
    ("Need earned leave next week Monday to Wednesday.", {"leave_type": "earned", "from_date": "next monday", "to_date": "next wednesday"}),
    ("Mark sick leave for me today, not well.", {"leave_type": "sick", "from_date": "today", "to_date": "today"}),
    ("Casual leave laga do 5th July ko.", {"leave_type": "casual", "from_date": "2026-07-05", "to_date": "2026-07-05"}),
    ("Apply my earned leave from 29th June to 2nd July.", {"leave_type": "earned", "from_date": "2026-06-29", "to_date": "2026-07-02"}),
]
# ──────────────────────────────────────────────
# GET_PAYSLIP examples
# ──────────────────────────────────────────────
get_payslip_examples = [
    ("Can I get my payslip for April 2026?", {"month": "April", "year": "2026"}),
    ("Send me my May 2026 salary slip.", {"month": "May", "year": "2026"}),
    ("I need my payslip for last month.", {"month": "last month", "year": "last month"}),
    ("Show me my March 2026 salary slip.", {"month": "March", "year": "2026"}),
    ("Download payslip for February 2026.", {"month": "February", "year": "2026"}),
    ("yaar April 2026 ki payslip bhej do", {"month": "April", "year": "2026"}),
    ("I want my salary breakdown for May 2026.", {"month": "May", "year": "2026"}),
    ("Get me the January 2026 payslip please.", {"month": "January", "year": "2026"}),
    ("Payslip for December 2025 needed.", {"month": "December", "year": "2025"}),
    ("Pull up my salary slip for June 2026.", {"month": "June", "year": "2026"}),
    ("I need April 2026's payslip for my loan application.", {"month": "April", "year": "2026"}),
    ("Share my March 2026 salary details.", {"month": "March", "year": "2026"}),
    ("payslip do February 2026 ka", {"month": "February", "year": "2026"}),
    ("Could you send my November 2025 payslip?", {"month": "November", "year": "2025"}),
    ("I want to download my salary slip for May 2026.", {"month": "May", "year": "2026"}),
    ("Get my payslip for the month of April 2026.", {"month": "April", "year": "2026"}),
    ("Need salary slip - January 2026.", {"month": "January", "year": "2026"}),
    ("Please provide my payslip for February 2026.", {"month": "February", "year": "2026"}),
    ("Show salary breakdown for March 2026.", {"month": "March", "year": "2026"}),
    ("Can I see my June 2026 payslip?", {"month": "June", "year": "2026"}),
    ("I need last month's salary slip for my visa documents.", {"month": "last month", "year": "last month"}),
    ("Fetch my October 2025 payslip.", {"month": "October", "year": "2025"}),
    ("Salary slip for April 2026 please.", {"month": "April", "year": "2026"}),
    ("Give me my pay statement for May 2026.", {"month": "May", "year": "2026"}),
    ("I'd like my March 2026 payslip.", {"month": "March", "year": "2026"}),
    ("Email me my February 2026 salary slip.", {"month": "February", "year": "2026"}),
    ("Need my January 2026 salary breakdown.", {"month": "January", "year": "2026"}),
    ("payslip chahiye June 2026 ki", {"month": "June", "year": "2026"}),
    ("Pull my December 2025 pay slip.", {"month": "December", "year": "2025"}),
    ("I want to view my salary slip for April 2026.", {"month": "April", "year": "2026"}),
    ("Get me this month's payslip.", {"month": "this month", "year": "this month"}),
    ("Share my September 2025 salary details please.", {"month": "September", "year": "2025"}),
    ("My July 2025 payslip, can you fetch it?", {"month": "July", "year": "2025"}),
    ("salary slip June 2026 ka download karna hai", {"month": "June", "year": "2026"}),
    ("Need August 2025 payslip for ITR filing.", {"month": "August", "year": "2025"}),
    ("Give me last month's pay statement quickly.", {"month": "last month", "year": "last month"}),
]

# ──────────────────────────────────────────────
# RAISE_TICKET examples
# ──────────────────────────────────────────────
raise_ticket_examples = [
    ("My salary was short by 8000 this month, something's wrong.", {"category": "payroll", "description": "Salary short by 8000 this month"}),
    ("I haven't received my travel reimbursement yet.", {"category": "reimbursement", "description": "Travel reimbursement not received"}),
    ("My attendance shows absent on a day I worked.", {"category": "attendance", "description": "Attendance marked absent on a working day"}),
    ("My PF transfer has been pending for weeks and nobody is responding.", {"category": "pf", "description": "PF transfer pending with no response"}),
    ("There's a tax deduction error in my payslip.", {"category": "payroll", "description": "Incorrect tax deduction in payslip"}),
    ("I submitted my medical bills but reimbursement hasn't come through.", {"category": "reimbursement", "description": "Medical bill reimbursement not processed"}),
    ("My March salary was credited incorrectly.", {"category": "payroll", "description": "March salary credited incorrectly"}),
    ("I never received my February payslip, it's missing from the system.", {"category": "payroll", "description": "February payslip missing from system"}),
    ("My leave got deducted even though it was an approved work from home day.", {"category": "attendance", "description": "Leave wrongly deducted for approved WFH day"}),
    ("Reimbursement approved is less than what I claimed.", {"category": "reimbursement", "description": "Approved reimbursement less than claimed"}),
    ("HR isn't responding to my emails about my relieving letter.", {"category": "general", "description": "No HR response on relieving letter"}),
    ("My bonus wasn't included in this month's salary.", {"category": "payroll", "description": "Bonus missing from this month's salary"}),
    ("There's a mistake in my PF contribution amount.", {"category": "pf", "description": "Error in PF contribution amount"}),
    ("I was double-marked absent for last Friday, please fix.", {"category": "attendance", "description": "Double absence marked for last Friday"}),
    ("My internet reimbursement claim has been stuck for a month.", {"category": "reimbursement", "description": "Internet reimbursement claim stuck for a month"}),
    ("salary calculation galat hai is mahine, theek karo.", {"category": "payroll", "description": "Salary calculation incorrect this month"}),
    ("I need to report that my insurance wasn't activated.", {"category": "general", "description": "Insurance not activated"}),
    ("My overtime hours weren't paid this month.", {"category": "payroll", "description": "Overtime hours not paid this month"}),
    ("The reimbursement portal rejected my claim without any reason.", {"category": "reimbursement", "description": "Reimbursement claim rejected without reason"}),
    ("My UAN is linked incorrectly in the PF account.", {"category": "pf", "description": "UAN linked incorrectly in PF account"}),
    ("I want to complain about wrong attendance records for last week.", {"category": "attendance", "description": "Incorrect attendance records for last week"}),
    ("My salary slip shows a deduction I don't recognise.", {"category": "payroll", "description": "Unrecognised deduction in salary slip"}),
    ("Nobody processed my food allowance reimbursement.", {"category": "reimbursement", "description": "Food allowance reimbursement not processed"}),
    ("There's an issue with my gratuity calculation.", {"category": "payroll", "description": "Issue with gratuity calculation"}),
    ("My biometric isn't registering and my attendance is affected.", {"category": "attendance", "description": "Biometric not registering, attendance affected"}),
    ("I have a grievance about my appraisal not being recorded.", {"category": "general", "description": "Appraisal not recorded in system"}),
    ("My PF withdrawal request was rejected wrongly.", {"category": "pf", "description": "PF withdrawal request wrongly rejected"}),
    ("The HRA component in my salary looks wrong.", {"category": "payroll", "description": "HRA component appears incorrect"}),
    ("I raised a reimbursement two months ago and still nothing.", {"category": "reimbursement", "description": "Reimbursement pending for two months"}),
    ("My work from home days are being counted as leaves.", {"category": "attendance", "description": "WFH days counted as leaves"}),
    ("There's an error in my Form 16, I need it corrected.", {"category": "payroll", "description": "Error in Form 16 needs correction"}),
    ("I want to escalate that my onboarding documents are still not verified.", {"category": "general", "description": "Onboarding documents not verified"}),
    ("PF ka paisa transfer nahi hua abhi tak, complaint karni hai.", {"category": "pf", "description": "PF amount not transferred yet"}),
    ("My cab reimbursement for last month is missing.", {"category": "reimbursement", "description": "Cab reimbursement for last month missing"}),
    ("I got marked half-day wrongly on 2nd June.", {"category": "attendance", "description": "Half-day wrongly marked on 2nd June"}),
    ("My ID card access isn't working, please raise this.", {"category": "general", "description": "ID card access not working"}),
]
# ──────────────────────────────────────────────
# GET_COMPANY_POLICY examples
# ──────────────────────────────────────────────
get_company_policy_examples = [
    ("What is the work from home policy?", {"topic": "wfh_policy"}),
    ("How many days notice period do I need to serve?", {"topic": "notice_period"}),
    ("What's the reimbursement limit for travel?", {"topic": "reimbursement_policy"}),
    ("How many days of paternity leave does the company offer?", {"topic": "leave_policy"}),
    ("What is the maternity leave policy here?", {"topic": "leave_policy"}),
    ("What are the company's remote work rules?", {"topic": "wfh_policy"}),
    ("How much earned leave can I accumulate as per policy?", {"topic": "leave_policy"}),
    ("What's the policy on comp-off?", {"topic": "leave_policy"}),
    ("How many days notice before resigning?", {"topic": "notice_period"}),
    ("What are the rules for claiming internet reimbursement?", {"topic": "reimbursement_policy"}),
    ("What's the official late-coming and attendance policy?", {"topic": "attendance_policy"}),
    ("How does the company handle public holidays?", {"topic": "holiday_policy"}),
    ("What's the rule for carry forward of leaves?", {"topic": "leave_policy"}),
    ("WFH ke rules kya hain?", {"topic": "wfh_policy"}),
    ("What's the limit on meal allowance claims?", {"topic": "reimbursement_policy"}),
    ("How many sick leaves are credited per year as per policy?", {"topic": "leave_policy"}),
    ("What is the notice period during probation?", {"topic": "notice_period"}),
    ("Are there rules around minimum office days per week?", {"topic": "wfh_policy"}),
    ("How many work from home days are allowed per month?", {"topic": "wfh_policy"}),
    ("What documents are needed for reimbursement claims?", {"topic": "reimbursement_policy"}),
    ("Tell me about the maternity benefit rules.", {"topic": "leave_policy"}),
    ("What's the policy on working on public holidays?", {"topic": "holiday_policy"}),
    ("How long is the notice period for senior employees?", {"topic": "notice_period"}),
    ("Can I encash my unused earned leaves? What's the rule?", {"topic": "leave_policy"}),
    ("What is the relocation allowance limit?", {"topic": "reimbursement_policy"}),
    ("How many floating holidays do we get in a year?", {"topic": "holiday_policy"}),
    ("What's the grace period for late attendance?", {"topic": "attendance_policy"}),
    ("notice period kitna hota hai company mein?", {"topic": "notice_period"}),
    ("What is the list of holidays this year?", {"topic": "holiday_policy"}),
    ("How is attendance regularization handled?", {"topic": "attendance_policy"}),
    ("What's the cap on mobile bill reimbursement?", {"topic": "reimbursement_policy"}),
    ("Leave carry forward ka rule kya hai?", {"topic": "leave_policy"}),
    ("Can I work fully remote? What does the policy say?", {"topic": "wfh_policy"}),
    ("What's the notice period if I'm still in probation?", {"topic": "notice_period"}),
    ("How many restricted holidays can I take?", {"topic": "holiday_policy"}),
]



print(f"check_leave_balance_examples:   {len(check_leave_balance_examples)}")
print(f"apply_leave_examples: {len(apply_leave_examples)}")
print(f"get_payslip_examples:   {len(get_payslip_examples)}")
print(f"raise_ticket_examples: {len(raise_ticket_examples)}")
print(f"get_company_policy_examples:   {len(get_company_policy_examples)}")

print(f"Total: {len(check_leave_balance_examples) + len(apply_leave_examples)+ len(get_payslip_examples)+ len(raise_ticket_examples)+ len(get_company_policy_examples)}")

check_leave_balance_examples:   35
apply_leave_examples: 36
get_payslip_examples:   36
raise_ticket_examples: 36
get_company_policy_examples:   35
Total: 178


In [13]:
# Convert to the conversations format that SFTTrainer expects

def make_conversation(user_query, tool_name, params):
    tool_call = json.dumps({"tool": tool_name, "params": params})
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_query},
            {"role": "assistant", "content": tool_call},
        ]
    }

dataset_rows = []

for query, params in check_leave_balance_examples:
    dataset_rows.append(make_conversation(query, "check_leave_balance", params))
for query, params in apply_leave_examples:
    dataset_rows.append(make_conversation(query, "apply_leave", params))
for query, params in get_payslip_examples:
    dataset_rows.append(make_conversation(query, "get_payslip", params))
for query, params in raise_ticket_examples:
    dataset_rows.append(make_conversation(query, "raise_ticket", params))
for query, params in get_company_policy_examples:
    dataset_rows.append(make_conversation(query, "get_company_policy", params))


# Shuffle so the model doesn't just learn "first half = tool A"
random.seed(42)
random.shuffle(dataset_rows)

# Let's look at a few examples
print("Sample training conversation:")
print(json.dumps(dataset_rows[0], indent=2))
print("\n" + "─" * 60)
print(json.dumps(dataset_rows[1], indent=2))
print("\n" + "─" * 60)
print(json.dumps(dataset_rows[2], indent=2))
print("\n" + "─" * 60)
print(json.dumps(dataset_rows[3], indent=2))
print("\n" + "─" * 60)
print(json.dumps(dataset_rows[4], indent=2))

Sample training conversation:
{
  "conversations": [
    {
      "role": "system",
      "content": "You are a Human resource query support assistant. You have access to exactly five tools:\n\n1. check_leave_balance: Call this tool when the employee wants to know how many leaves they have remaining.\n   Triggers include questions about leave count, remaining days, available leaves, or balance for any leave type \u2014 casual, sick, or earned.\n   Parameters: {\"leave_type\": string}\n\n2. apply_leave: Call this tool when the employee wants to request or apply for a leave for specific dates.\n   Triggers include requests to book, apply, or take a leave on a particular day or date range.\n   Parameters: {\"leave_type\": string, \"from_date\": date , \"to_date\": date}\n\n3. get_payslip: Call this tool when the employee wants to view or download their salary slip for a specific month.\n   Triggers include mentions of payslip, salary slip, or salary details for a given month or year.\n   P

In [14]:
from datasets import Dataset

dataset = Dataset.from_list(dataset_rows)
print(dataset)
print(f"\nTotal training examples: {len(dataset)}")

Dataset({
    features: ['conversations'],
    num_rows: 178
})

Total training examples: 178


---
## Step 4: Attach LoRA Adapters

Instead of updating all 1B parameters, LoRA trains two small matrices (rank 16) on each attention and MLP layer.

**Why this matters:** A 1B model has ~1 billion weights. With LoRA rank=16, we only train ~10 million weights (1%). This means:
- 50-70% less VRAM
- 2x faster training
- The base model knowledge is preserved — we're just teaching it a new output format

In [15]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # LoRA rank — 16 is a good default
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention layers -> Q, K, V, O
        "gate_proj", "up_proj", "down_proj",     # FFNN layers -> Activation Function
    ],
    lora_alpha = 16,     # 16/16 = 2, Scaling factor of lora updates: If you set alpha=32 with r=16, the LoRA updates are multiplied by 2x. The model learns faster but can overshoot and destabilize.
    lora_dropout = 0,          # how many neurons are ignored
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # 30% less VRAM
    random_state = 42,
    max_seq_length = 2048,
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Trainable parameters: 11,272,192 / 760,547,328 (1.48%)


---
## Step 5: Fine-Tune!

We train for **3 epochs** over 160 examples = 480 training steps.
On a T4, this takes roughly **5–10 minutes**.

In [16]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
formatted_rows = []
for row in dataset_rows:
    text = tokenizer.apply_chat_template(
        row["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    formatted_rows.append({"text": text})

dataset = Dataset.from_list(formatted_rows)

# Check what the training text actually looks like
print(dataset[0]["text"][:500])
print("─" * 60)
print(f"Total training examples: {len(dataset)}")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",         # <-- just point to the pre-formatted column
        per_device_train_batch_size = 8, # 4 GPUs -> 16 examples together = loss16 , next step loss16, update gradients
        gradient_accumulation_steps = 2,
        num_train_epochs = 5, # Number of times we are going to go over the same training set, 80 examples (Run1, Run2, Run3)
        warmup_steps = 5,
        learning_rate = 2e-4,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        weight_decay = 0.01,
        seed = 42,
        output_dir = "tool-routing-checkpoints",
        max_seq_length = 2048,
    ),
)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 05 Jun 2026

You are a Human resource query support assistant. You have access to exactly five tools:

1. check_leave_balance: Call this tool when the employee wants to know how many leaves they have remaining.
   Triggers include questions about leave count, remaining days, available leaves, or balance for any leave type — casual, sick, or earned.
   Parameters: {"leave_type": string}

────────────────────────────────────────────────────────────
Total training examples: 178


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/178 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [17]:
print("Starting training...")
trainer_stats = trainer.train()

print(f"\n{'=' * 50}")
print(f"Training complete!")
print(f"Total time:   {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"Final loss:   {trainer_stats.metrics['train_loss']:.4f}")
print(f"{'=' * 50}")

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 178 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.584712
2,2.563139
3,2.518536
4,2.404063
5,2.226994
6,2.041080
7,1.874430
8,1.651933
9,1.491027
10,1.291610


Unsloth: Restored added_tokens_decoder metadata in tool-routing-checkpoints/checkpoint-60/tokenizer_config.json.



Training complete!
Total time:   204 seconds
Final loss:   0.5249


---
## Step 6: Test the Fine-Tuned Model

Now the moment of truth. We run the **exact same queries** from Step 2.

In [18]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

print("=" * 70)
print("FINE-TUNED MODEL RESPONSES")
print("=" * 70)

correct = 0
for query, expected_tool in zip(TEST_QUERIES, EXPECTED):
    response = run_inference(model, tokenizer, query)
    print(f"\n{'─' * 60}")
    print(f"QUERY:    {query[:80]}")
    print(f"EXPECTED: {expected_tool}")
    print(f"GOT:      {response[:200]}")

    try:
        parsed = json.loads(response)
        if parsed.get("tool") == expected_tool:
            print("RESULT:   ✅ Correct tool + valid JSON")
            correct += 1
        else:
            print(f"RESULT:   ❌ Wrong tool (got '{parsed.get('tool')}')'")
    except json.JSONDecodeError:
        print("RESULT:   ❌ Not valid JSON")

print(f"\n{'=' * 70}")
print(f"Score: {correct}/{len(TEST_QUERIES)} correct")
print(f"{'=' * 70}")

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FINE-TUNED MODEL RESPONSES


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    Forget the balance, just lock in a casual leave for me this Friday.
EXPECTED: apply_leave
GOT:      {"tool": "apply_leave", "params": {"leave_type": "casual", "from_date": "this Friday", "to_date": "this Friday"}}
RESULT:   ✅ Correct tool + valid JSON


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


────────────────────────────────────────────────────────────
QUERY:    Apply casual leave for 25th June for me, and remind me what the holiday list loo
EXPECTED: apply_leave
GOT:      {"tool": "apply_leave", "params": {"leave_type": "casual", "from_date": "2026-06-25", "to_date": "2026-06-25"}}
RESULT:   ✅ Correct tool + valid JSON


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    If I forget to punch in, how does attendance regularization work here?
EXPECTED: get_company_policy
GOT:      {"tool": "get_company_policy", "params": {"topic": "attendance_policy"}}
RESULT:   ✅ Correct tool + valid JSON


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    I was physically in office on 3rd June but the system says I was absent.
EXPECTED: raise_ticket
GOT:      {"tool": "raise_ticket", "params": {"category": "general", "description": "System says I was absent when I was physically in office"}}
RESULT:   ✅ Correct tool + valid JSON

────────────────────────────────────────────────────────────
QUERY:    Pull my November 2025 pay statement, I need it for tax filing.
EXPECTED: get_payslip
GOT:      {"tool": "get_payslip", "params": {"month": "November", "year": "2025"}}
RESULT:   ✅ Correct tool + valid JSON

Score: 18/18 correct


---
## Step 7: Try Completely New Queries

These are queries the model has **never seen** — testing generalization.

In [19]:
NEW_QUERIES = [
    # Should → check_leave_balance
    ("How many casual leaves are still with me?", "check_leave_balance"),
    ("Sick leave balance batao zara", "check_leave_balance"),
    ("Do I have any earned leaves left to use?", "check_leave_balance"),
    ("What is my overall leave balance right now?", "check_leave_balance"),
    ("Any casual leaves remaining for me?", "check_leave_balance"),

    # Should → apply_leave
    ("Apply a casual leave for me on 17th June.", "apply_leave"),
    ("I am sick, put a sick leave for today.", "apply_leave"),
    ("Book earned leave from 4th to 6th August.", "apply_leave"),
    ("Need leave next Tuesday, casual one.", "apply_leave"),
    ("kal sick leave laga dena please", "apply_leave"),

    # Should → get_payslip
    ("Can you fetch my payslip for July 2026?", "get_payslip"),
    ("I want last month salary slip.", "get_payslip"),
    ("Send October 2025 pay statement for me.", "get_payslip"),
    ("payslip chahiye May 2026 ki", "get_payslip"),
    ("Download my salary slip for September 2025.", "get_payslip"),

    # Should → raise_ticket
    ("My salary got credited less than usual this month.", "raise_ticket"),
    ("Still waiting on my laptop reimbursement, please act.", "raise_ticket"),
    ("I was marked absent on a day I clearly worked.", "raise_ticket"),
    ("My PF account is not reflecting last two contributions.", "raise_ticket"),
    ("No one from HR has responded to my query for days.", "raise_ticket"),

    # Should → get_company_policy
    ("What is the policy on carrying forward unused leaves?", "get_company_policy"),
    ("How many remote days are allowed in a month?", "get_company_policy"),
    ("What notice period applies after probation?", "get_company_policy"),
    ("What is the cap on travel reimbursement?", "get_company_policy"),
    ("How many public holidays do we get this year?", "get_company_policy"),

]

print("=" * 70)
print("GENERALIZATION TEST — QUERIES NEVER SEEN IN TRAINING")
print("=" * 70)

correct = 0
for query, expected_tool in NEW_QUERIES:
    response = run_inference(model, tokenizer, query)
    print(f"\n{'─' * 60}")
    print(f"QUERY:    {query}")
    print(f"EXPECTED: {expected_tool}")
    print(f"GOT:      {response[:200]}")

    try:
        parsed = json.loads(response)
        tool_correct = parsed.get("tool") == expected_tool
        has_params = "params" in parsed
        if tool_correct and has_params:
            print(f"RESULT:   ✅ Correct tool + valid params")
            correct += 1
        elif tool_correct:
            print(f"RESULT:   ⚠️  Correct tool but missing params")
        else:
            print(f"RESULT:   ❌ Wrong tool")
    except json.JSONDecodeError:
        print(f"RESULT:   ❌ Not valid JSON")

print(f"\n{'=' * 70}")
print(f"Generalization score: {correct}/{len(NEW_QUERIES)} correct")
print(f"{'=' * 70}")

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERALIZATION TEST — QUERIES NEVER SEEN IN TRAINING


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    How many casual leaves are still with me?
EXPECTED: check_leave_balance
GOT:      {"tool": "check_leave_balance", "params": {"leave_type": "casual"}}
RESULT:   ✅ Correct tool + valid params


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


────────────────────────────────────────────────────────────
QUERY:    How many remote days are allowed in a month?
EXPECTED: get_company_policy
GOT:      {"tool": "get_company_policy", "params": {"topic": "leave_policy"}}
RESULT:   ✅ Correct tool + valid params


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    What notice period applies after probation?
EXPECTED: get_company_policy
GOT:      {"tool": "get_company_policy", "params": {"topic": "notice_period"}}
RESULT:   ✅ Correct tool + valid params


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    What is the cap on travel reimbursement?
EXPECTED: get_company_policy
GOT:      {"tool": "get_company_policy", "params": {"topic": "reimbursement_policy"}}
RESULT:   ✅ Correct tool + valid params

────────────────────────────────────────────────────────────
QUERY:    How many public holidays do we get this year?
EXPECTED: get_company_policy
GOT:      {"tool": "get_company_policy", "params": {"topic": "public_holidays"}}
RESULT:   ✅ Correct tool + valid params

Generalization score: 25/25 correct


---
## Step 8: Save & Export

Export to GGUF for local deployment with Ollama or llama.cpp.

In [20]:
# Save LoRA adapters (small, ~30MB)
model.save_pretrained("tool-router-lora")
tokenizer.save_pretrained("tool-router-lora")
print("LoRA adapters saved to tool-router-lora/")

# Optional: Export merged GGUF for local deployment
# Uncomment the line below if you want to run this with Ollama
# model.save_pretrained_gguf("tool-router-gguf", tokenizer, quantization_method="q4_k_m")
# print("GGUF exported to tool-router-gguf/")

Unsloth: Restored added_tokens_decoder metadata in tool-router-lora/tokenizer_config.json.


LoRA adapters saved to tool-router-lora/
